# BART-large CNN — DIMER E2E abstractive summarization fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/bart-cnn-summarization-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/bart-cnn-summarization-pipeline/blob/main/tutorials/bart_summarization_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fbart--large--cnn-ffcc4d?style=flat)](https://huggingface.co/facebook/bart-large-cnn) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Ffairseq-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/fairseq/tree/main/examples/bart) [![arXiv](https://img.shields.io/badge/arXiv-1910.13461-b31b1b.svg)](https://arxiv.org/abs/1910.13461)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** abstractive summarization of one English document by deterministic beam search (pinned generation defaults; token counts and a truncation flag reported; no score) and bounded supervised fine-tuning of the last decoder blocks on a referenced summarization corpus, using the pinned `facebook/bart-large-cnn` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/bart_summarization_pipeline/`, at revision `2d891a53fb98`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `37f520fa929c961707657b28798b30c003dd100b` (~1628 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned BART-large CNN snapshot (safetensors, 1.6 GB), fetches the three digest-pinned SciTLDR-A files from the project repository (5.5 MB, no credential), filters and draws 300 / 50 / 100 training, validation and test abstracts from the release's own paper-disjoint members, summarises one synthetic news-style document through the inference contract with the pinned generation defaults and a rejection probe, scores the frozen model on the test abstracts with ROUGE-1/2/L under TL;DR-length settings beside the Lead-1 and Lead-3 baselines, runs a bounded fine-tuning of the last two decoder blocks on the training abstracts with validation-ROUGE-L epoch selection, scores the held-out split again, summarises new abstracts with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify summary parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about twelve minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own document–summary pairs as a CSV (columns `id`, `source`, `target`), a JSON array or a JSONL file of `{{id, source, target}}` or `{{id, source, targets: [...]}}` records. They pass through the same validation, seeded source-disjoint split, baselines, fine-tuning, held-out evaluation, inference, artifact export and reload-parity cells as the SciTLDR sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference the byte-level BPE tokenizer encodes the whole document once, the 12-layer bidirectional encoder of the 406 M-parameter BART-large reads it, and the 12-layer autoregressive decoder — fine-tuned upstream on CNN/DailyMail article/highlight pairs — writes a summary token by token under **deterministic beam search**: `num_beams` 4, `length_penalty` 2.0, `no_repeat_ngram_size` 3, `early_stopping`, and the length bounds of the snapshot's `generation_config_for_summarization.json` (`max_length` 142 / `min_length` 56, exposed as `DEFAULT_MAX_NEW_TOKENS` 141 / `DEFAULT_MIN_NEW_TOKENS` 55 because the upstream bounds count the decoder start token). What the upstream checkpoint supplies is the encoder-decoder, the language-model head, the generation config and the tokenizer; what the carried pipeline module adds is manifest verification, input and generation-setting validation against named ceilings (over-long documents are rejected, not truncated or chunked), a fixed output contract that reports `generated_tokens`, `input_tokens`, `truncated` and `stopped_by`, and the `validate_inputs` and `evaluation_report` stage helpers. **The pipeline emits no score, probability or quality metric** — a summary is free text.

What this notebook adds to inference is **adaptation with references**. The dataset is real and deliberately **out of domain**: SciTLDR-A (Cachola et al., 2020; Apache-2.0), one-sentence TL;DR summaries of computer-science paper abstracts — three digest-pinned JSON-Lines files fetched from the project repository at a pinned commit. A news summariser writes three-sentence, largely extractive highlights here; the fine-tuning question is whether a bounded adaptation of the last two decoder blocks moves it to the short TL;DR register on held-out papers. Three metrics are implemented in the carried `metrics.py` (corpus **ROUGE-1/2/L** F1, best over the references, rouge-score-style, not rouge-score-identical), and two **Lead baselines** — the first sentence and the first three sentences of the abstract as the summary — show where a system that does no modelling sits. Nothing here is a quality claim about your documents: it is one seeded split of one corpus.

**Length note:** the frozen model's pinned news defaults force at least 55 new tokens, three times a TL;DR; every ROUGE number in this notebook is therefore produced under explicit TL;DR-length settings (`max_new_tokens` 48, `min_new_tokens` 0, `num_beams` 4) applied identically to the frozen and the adapted model, and the inference contract in Section 5 shows the pinned defaults on a news-style document first.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, dataset and metrics modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned referenced summarization corpus and validate and split it without leakage; summarise through the public API with the pinned generation defaults and read `generated_tokens`, `truncated` and `stopped_by` correctly; score the frozen model against references beside two Lead baselines and read why summary length drives ROUGE; run a bounded fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on an independent test split; summarise new abstracts; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** extractive summarization with guaranteed source spans, multi-document or long-document (chunked) summarization, headline generation, sampling-based or diverse decoding, full-model or encoder fine-tuning, classification or question answering (the `bart-mnli-zero-shot-classification-pipeline` sibling covers zero-shot classification), non-English text, any faithfulness or factuality score, and any claim that a SciTLDR split stands in for your documents. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is adequate but slow for a 406 M-parameter decoder: the build record measured 6 s to load and digest-verify the 1.6 GB snapshot, about 1.8 s per abstract for 4-beam TL;DR-length generation (three minutes for the 100-abstract test split) and about 40 s per training epoch over 300 abstracts plus a 50-abstract validation pass per epoch. The pinned `torch==2.14.0` install and the 1.6 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what an encoder-decoder (seq2seq) model is; what beam search does and why it is deterministic; why an abstractive summary can state things the source does not (hallucination); what ROUGE measures and why it is neither faithfulness nor a human judgement.
- **Data contract:** records are `{{id, source, targets}}` — a document and one or more reference summaries (`{{id, source, target}}` with a single string is accepted and normalised), the source 1..40,000 characters and at most 1,024 BPE tokens at inference, each reference 1..2,000 characters, ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a dataset needs 8..20,000 records; sources are de-duplicated case-insensitively before splitting so the same document never sits in two splits; during training only, sources are truncated to 512 and targets to 64 BPE tokens (inference never truncates — it rejects). BYOD accepts CSV, JSON or JSONL in that shape.
- **Validation is structural, not semantic:** nothing checks that a reference is a faithful summary of its source or that a source is prose — a mislabelled corpus is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an internal report archive with its executive summaries is exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches three pinned objects (`train.jsonl` 3,155,015 bytes, `dev.jsonl` 1,124,865 bytes, `test.jsonl` 1,204,107 bytes; SHA-256 `b222771d…` / `3191fa98…` / `fb42dd6c…`) from `raw.githubusercontent.com` at the pinned `allenai/scitldr` commit over HTTPS, each refused on any mismatch before it is read; SciTLDR is Apache-2.0 (Cachola et al., 2020).
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/bart-large-cnn` snapshot (~1628 MB in total) at revision `37f520fa929c…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'bart-cnn-summarization-pipeline',
    'repository_revision': '2d891a53fb98773068dd68f8d60d4af3dae78cea',
    'embedded_module': 'src/bart_summarization_pipeline/pipeline.py',
    'embedded_modules': ['src/bart_summarization_pipeline/metrics.py', 'src/bart_summarization_pipeline/pipeline.py', 'src/bart_summarization_pipeline/samples.py'],
    'module_sha256': 'b83fbebd79bdf47b24511135c4b5e8b734bd18a4822e4909b420f505648c1360',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/bart_summarization_pipeline/` @ `2d891a53fb98`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/bart_summarization_pipeline/metrics.py`

In [ ]:
"""Reference-based summarization metrics (ROUGE-1/2/L F1, own implementation) and the Lead baseline.

ROUGE follows the `rouge-score` package's recipe without stemming: lower-case, keep runs of ASCII letters
and digits as tokens, unigram/bigram overlap F1 for ROUGE-1/2, longest-common-subsequence F1 for ROUGE-L
(sentence-level, the whole summary as one sequence). With several references per document the best score
over the references is taken (the SciTLDR convention), then averaged over documents and reported in percent.
Values are close to, but not identical with, `rouge-score` — no stemming, no bootstrap — and neither is a
human judgement of faithfulness.
"""

from __future__ import annotations

import re
from collections import Counter
from collections.abc import Mapping, Sequence
from typing import Any

_TOKEN_RE = re.compile(r"[a-z0-9]+")
_SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")
METRIC_DEFINITIONS = {
    "rouge1": (
        "unigram overlap F1 between the summary and the best-matching reference, averaged over documents; "
        "percent"
    ),
    "rouge2": (
        "bigram overlap F1 between the summary and the best-matching reference, averaged over documents; "
        "percent"
    ),
    "rougeL": (
        "longest-common-subsequence F1 (whole summary as one token sequence) against the best-matching "
        "reference, averaged over documents; percent"
    ),
    "tokenisation": (
        "lower-cased runs of ASCII letters and digits; no stemming; "
        "rouge-score-style, not rouge-score-identical"
    ),
}


def rouge_tokens(text: str) -> list[str]:
    return _TOKEN_RE.findall(text.lower())


def _f1(overlap: int, n_hyp: int, n_ref: int) -> float:
    if overlap == 0 or n_hyp == 0 or n_ref == 0:
        return 0.0
    precision, recall = overlap / n_hyp, overlap / n_ref
    return 2 * precision * recall / (precision + recall)


def rouge_n(hypothesis: str, reference: str, n: int) -> float:
    hyp = rouge_tokens(hypothesis)
    ref = rouge_tokens(reference)
    hyp_grams = Counter(tuple(hyp[i : i + n]) for i in range(len(hyp) - n + 1))
    ref_grams = Counter(tuple(ref[i : i + n]) for i in range(len(ref) - n + 1))
    overlap = sum((hyp_grams & ref_grams).values())
    return _f1(overlap, sum(hyp_grams.values()), sum(ref_grams.values()))


def _lcs_length(a: Sequence[str], b: Sequence[str]) -> int:
    if not a or not b:
        return 0
    previous = [0] * (len(b) + 1)
    for token in a:
        current = [0]
        for j, other in enumerate(b, start=1):
            current.append(previous[j - 1] + 1 if token == other else max(previous[j], current[j - 1]))
        previous = current
    return previous[-1]


def rouge_l(hypothesis: str, reference: str) -> float:
    hyp = rouge_tokens(hypothesis)
    ref = rouge_tokens(reference)
    return _f1(_lcs_length(hyp, ref), len(hyp), len(ref))


def rouge_scores(hypothesis: str, references: Sequence[str]) -> dict[str, float]:
    """Best ROUGE-1/2/L F1 over the references for one summary (fractions in 0..1)."""
    if not references:
        raise ValueError("at least one reference is required")
    return {
        "rouge1": max(rouge_n(hypothesis, r, 1) for r in references),
        "rouge2": max(rouge_n(hypothesis, r, 2) for r in references),
        "rougeL": max(rouge_l(hypothesis, r) for r in references),
    }


def summary_metrics(hypotheses: Sequence[str], references: Sequence[Sequence[str]]) -> dict[str, Any]:
    """Corpus ROUGE-1/2/L F1 in percent over parallel summaries and reference lists."""
    if len(hypotheses) != len(references):
        raise ValueError(f"{len(hypotheses)} summaries but {len(references)} reference lists")
    if not hypotheses:
        raise ValueError("no summaries to score")
    per_doc = [rouge_scores(h, r) for h, r in zip(hypotheses, references, strict=True)]
    return {
        "n": len(hypotheses),
        "rouge1": 100.0 * sum(d["rouge1"] for d in per_doc) / len(per_doc),
        "rouge2": 100.0 * sum(d["rouge2"] for d in per_doc) / len(per_doc),
        "rougeL": 100.0 * sum(d["rougeL"] for d in per_doc) / len(per_doc),
        "mean_summary_words": sum(len(h.split()) for h in hypotheses) / len(hypotheses),
        "mean_reference_words": sum(len(r[0].split()) for r in references) / len(references),
        "definitions": dict(METRIC_DEFINITIONS),
    }


def lead_sentences(text: str, n_sentences: int = 1) -> str:
    """The first `n_sentences` sentences of a document (a naive `.!?` split)."""
    if n_sentences < 1:
        raise ValueError("n_sentences must be at least 1")
    return " ".join(_SENTENCE_RE.split(text.strip())[:n_sentences])


def lead_baseline(records: Sequence[Mapping[str, Any]], *, n_sentences: int = 1) -> dict[str, Any]:
    """Lead-N: the first N sentences of the source submitted as the summary — the classic extractive floor."""
    result = summary_metrics(
        [lead_sentences(r["source"], n_sentences) for r in records], [r["targets"] for r in records]
    )
    result["baseline"] = (
        f"lead-{n_sentences} (the first {n_sentences} sentence(s) of the source as the summary)"
    )
    return result

**Module 2/3:** `src/bart_summarization_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Abstractive summarization with the pinned ``facebook/bart-large-cnn`` checkpoint.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. One task method, ``summarize``: deterministic beam search
whose defaults are the values in the snapshot's ``generation_config_for_summarization.json``.

The adaptation contract (``evaluate``, ``adapt``, ``save_artifact``, ``from_artifact``) fine-tunes the last
decoder blocks on a validated ``{id, source, targets}`` dataset with validation-ROUGE-L epoch selection and
exports the trained tensors as a safetensors adapter bound to the pinned base weights. The inference contract
above is unchanged by it.
"""

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "facebook/bart-large-cnn"
MODEL_REVISION = "37f520fa929c961707657b28798b30c003dd100b"
MODEL_LICENSE = "mit"
MODEL_KEY = "bart-large-cnn"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. 1024 is max_position_embeddings in the snapshot config.json; longer inputs are rejected, not cut.
MAX_INPUT_TOKENS = 1024
MAX_TEXT_CHARS = 40_000  # pre-tokenisation guard on the input string; ~4 chars per BPE token on English text
MAX_NEW_TOKENS = 512  # ceiling on decoder steps per call
MAX_NUM_BEAMS = 8
MAX_NO_REPEAT_NGRAM_SIZE = 10
LENGTH_PENALTY_RANGE = (-5.0, 5.0)
# Defaults read from the snapshot's generation_config_for_summarization.json (identical to
# generation_config.json): num_beams 4, length_penalty 2.0, no_repeat_ngram_size 3, early_stopping true,
# max_length 142, min_length 56. Upstream's max_length/min_length count the decoder start token, so the
# equivalent *new*-token bounds are one lower.
DEFAULT_NUM_BEAMS = 4
DEFAULT_LENGTH_PENALTY = 2.0
DEFAULT_NO_REPEAT_NGRAM_SIZE = 3
DEFAULT_MAX_NEW_TOKENS = 141  # pinned max_length 142 - 1
DEFAULT_MIN_NEW_TOKENS = 55  # pinned min_length 56 - 1
EARLY_STOPPING = True
DECISION_RULE = (
    "deterministic beam search (do_sample=False, early_stopping=True): the highest length-penalised "
    "log-probability beam is returned; no sampling, no seed"
)
GENERATION_CONFIG_FILE = "generation_config_for_summarization.json"
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "40041830399afb5348525ef8354b007ecec4286fdf3524f7e6b54377e17096cb"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 406_290_432
DECODER_LAYERS = 12  # config.json decoder_layers
DEFAULT_TRAINABLE_DECODER_LAYERS = 2  # the last two decoder blocks (33,593,344 parameters)
MAX_TRAIN_SOURCE_TOKENS = 512  # source truncation ceiling during adaptation (never at inference)
MAX_TRAIN_TARGET_TOKENS = 64  # target truncation ceiling during adaptation
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.bart-large-cnn.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _check_settings(
    max_new_tokens: Any, min_new_tokens: Any, num_beams: Any, length_penalty: Any, no_repeat_ngram_size: Any
) -> dict[str, Any]:
    """Raise TypeError/ValueError naming the first violated generation ceiling; return the settings."""
    for name, value, low, high in (
        ("max_new_tokens", max_new_tokens, 1, MAX_NEW_TOKENS),
        ("min_new_tokens", min_new_tokens, 0, MAX_NEW_TOKENS),
        ("num_beams", num_beams, 1, MAX_NUM_BEAMS),
        ("no_repeat_ngram_size", no_repeat_ngram_size, 0, MAX_NO_REPEAT_NGRAM_SIZE),
    ):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} must be an int")
        if not low <= value <= high:
            raise ValueError(f"{name} must be between {low} and {high}, got {value}")
    if min_new_tokens > max_new_tokens:
        raise ValueError(f"min_new_tokens {min_new_tokens} exceeds max_new_tokens {max_new_tokens}")
    if isinstance(length_penalty, bool) or not isinstance(length_penalty, int | float):
        raise TypeError("length_penalty must be a float")
    if not LENGTH_PENALTY_RANGE[0] <= length_penalty <= LENGTH_PENALTY_RANGE[1]:
        raise ValueError(f"length_penalty must be within {LENGTH_PENALTY_RANGE}, got {length_penalty}")
    return {
        "max_new_tokens": max_new_tokens,
        "min_new_tokens": min_new_tokens,
        "num_beams": num_beams,
        "length_penalty": float(length_penalty),
        "no_repeat_ngram_size": no_repeat_ngram_size,
        "early_stopping": EARLY_STOPPING,
        "do_sample": False,
        "decision_rule": DECISION_RULE,
    }


def _check_text(text: Any, name: str = "text") -> str:
    if not isinstance(text, str):
        raise TypeError(f"{name} must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError(f"{name} is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"{name} has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    return text


def _check_input_tokens(n_input: int) -> int:
    """The encoder-token ceiling, applied once the tokenizer has counted."""
    if n_input > MAX_INPUT_TOKENS:
        raise ValueError(f"input is {n_input} tokens; ceiling is MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}")
    return n_input


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one non-empty str: the document to summarise (English news-style prose)",
    "text_chars": [1, MAX_TEXT_CHARS],
    "input_tokens": [1, MAX_INPUT_TOKENS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "min_new_tokens": [0, MAX_NEW_TOKENS],
    "num_beams": [1, MAX_NUM_BEAMS],
    "length_penalty": list(LENGTH_PENALTY_RANGE),
    "no_repeat_ngram_size": [0, MAX_NO_REPEAT_NGRAM_SIZE],
    "defaults_from": GENERATION_CONFIG_FILE,
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "byte-level BPE encoding with <s>/</s> added and no truncation: an input over MAX_INPUT_TOKENS is "
        "rejected with a ValueError naming the count, never cut"
    ),
}


def validate_inputs(
    texts: Sequence[str],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    min_new_tokens: int = DEFAULT_MIN_NEW_TOKENS,
    num_beams: int = DEFAULT_NUM_BEAMS,
    length_penalty: float = DEFAULT_LENGTH_PENALTY,
    no_repeat_ngram_size: int = DEFAULT_NO_REPEAT_NGRAM_SIZE,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``summarize`` would: both route through ``_check_text``
    and ``_check_settings``. ``summarize`` takes one text per call, so ``texts`` is the batch the notebook
    will loop over and every entry is validated with the same settings. The encoder-token ceiling
    (``MAX_INPUT_TOKENS``) needs the loaded tokenizer and is enforced inside ``summarize``.
    """
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a sequence of str, not a single string")
    if not texts:
        raise ValueError("texts must hold at least one item")
    checked = [_check_text(text, f"texts[{i}]") for i, text in enumerate(texts)]
    settings = _check_settings(
        max_new_tokens, min_new_tokens, num_beams, length_penalty, no_repeat_ngram_size
    )
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[i] if names else f"doc-{i}",
                "chars": len(text),
                "paragraphs": len(text.split("\n\n")),
            }
            for i, text in enumerate(checked)
        ],
        "generation": settings,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], references: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report for one ``summarize`` result.

    With ``references`` (one or more reference summaries for the same document) the report carries the
    ROUGE-1/2/L F1 of that single summary against its best-matching reference (`metrics.py`) with the
    verdict ``sample-sanity`` — one document is a plumbing check, not a quality measurement; the corpus-level
    stage is ``BARTSummarizationPipeline.evaluate``. Without references the verdict is ``not-measurable``.
    """
    pass  # standalone rewrite (build_notebook.py): `from .metrics import rouge_scores` removed — names are kernel globals defined by the carried modules

    generation = result.get("generation", {})
    supplied = references is not None
    metrics: list[dict[str, Any]] = []
    if supplied:
        refs = [str(r) for r in references if str(r).strip()]
        if not refs:
            raise ValueError("references must hold at least one non-empty summary")
        scores = rouge_scores(str(result.get("summary", "")), refs)
        metrics = [
            {"id": name, "value": 100.0 * value, "estimation": "single document, best of the references"}
            for name, value in scores.items()
        ]
    return {
        "task": "abstractive summarization of one English document (CNN/DailyMail fine-tune)",
        "score_semantics": (
            "the pipeline emits no probability, confidence or score: generated_tokens, input_tokens, "
            "truncated and stopped_by are counts and flags, and "
            f"{generation.get('decision_rule', DECISION_RULE)} produces some token at every step with no "
            "minimum-probability cut-off and no shipped acceptance threshold; ROUGE, when references are "
            "supplied, is n-gram agreement with those references (own implementation), not faithfulness"
        ),
        "sample_kind": sample_kind,
        "n_generated_tokens": int(result.get("generated_tokens", 0)),
        "truncated": bool(result.get("truncated", False)),
        "metrics": metrics,
        "baselines": [],
        "verdict": "sample-sanity" if supplied else "not-measurable",
        "reason": (
            "ROUGE-1/2/L are computed for one document against its reference summaries with the repository's "
            "own implementation; a single document states no dispersion and is not a quality measurement"
            if supplied
            else "no reference summary was supplied, so ROUGE cannot be computed; "
            "a summary has no ground truth here"
        ),
        "needs": (
            "one or more reference summaries per document from the deployment domain over enough documents "
            "to state a dispersion, scored with `evaluate` (ROUGE-1/2/L, own implementation; the upstream "
            "card reports ROUGE on CNN/DailyMail and nothing here reproduces it), plus a faithfulness check "
            "against "
            "the source, excluding or re-running outputs whose truncated flag is true; no proxy such as "
            "compression ratio or copy rate substitutes for that"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class BARTSummarizationPipeline:
    """``_runner(text, settings)`` -> ``(summary_text, generated_tokens, stopped_by)``;
    ``_count_tokens(text)`` -> encoder token count incl. <s>/</s>. Both injectable so tests run offline."""

    _runner: Callable[[str, dict[str, Any]], tuple[str, int, str]]
    _count_tokens: Callable[[str], int]
    device: str = "cpu"
    source: str = "injected"
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BARTSummarizationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), {"local_files_only": True}, "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, {}, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import BartForConditionalGeneration, BartTokenizerFast

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = BartTokenizerFast.from_pretrained(
            location, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = BartForConditionalGeneration.from_pretrained(
            location, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()
        special = {tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id}

        def count_tokens(text: str) -> int:
            return len(tokenizer(text, truncation=False)["input_ids"])

        def runner(text: str, settings: dict[str, Any]) -> tuple[str, int, str]:
            enc = tokenizer(text, return_tensors="pt", truncation=False).to(resolved_device)
            with torch.inference_mode():
                out = model.generate(
                    **enc,
                    max_new_tokens=settings["max_new_tokens"],
                    min_new_tokens=settings["min_new_tokens"],
                    num_beams=settings["num_beams"],
                    length_penalty=settings["length_penalty"],
                    no_repeat_ngram_size=settings["no_repeat_ngram_size"],
                    early_stopping=settings["early_stopping"],
                    do_sample=False,
                )
            ids = out[0].tolist()
            content = [t for t in ids[1:] if t not in special]  # ids[0] is the decoder start token
            stopped_by = "eos" if tokenizer.eos_token_id in ids[1:] else "max_new_tokens"
            return tokenizer.decode(content, skip_special_tokens=True).strip(), len(content), stopped_by

        return cls(runner, count_tokens, resolved_device, source, _model=model, _tokenizer=tokenizer)

    def summarize(
        self,
        text: str,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        min_new_tokens: int = DEFAULT_MIN_NEW_TOKENS,
        num_beams: int = DEFAULT_NUM_BEAMS,
        length_penalty: float = DEFAULT_LENGTH_PENALTY,
        no_repeat_ngram_size: int = DEFAULT_NO_REPEAT_NGRAM_SIZE,
    ) -> dict[str, Any]:
        """Summarise one document by deterministic beam search; defaults are the pinned generation config."""
        text = _check_text(text)
        settings = _check_settings(
            max_new_tokens, min_new_tokens, num_beams, length_penalty, no_repeat_ngram_size
        )
        n_input = _check_input_tokens(self._count_tokens(text))
        summary, n_generated, stopped_by = self._runner(text, settings)
        if not (isinstance(summary, str) and isinstance(n_generated, int) and isinstance(stopped_by, str)):
            raise RuntimeError("runner must return (str, int, str)")
        return {
            "summary": summary,
            "generated_tokens": n_generated,
            "input_tokens": n_input,
            "truncated": stopped_by != "eos",
            "stopped_by": stopped_by,
            "generation": settings,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def evaluate(self, records: Sequence[Mapping[str, Any]], **generation: Any) -> dict[str, Any]:
        """Summarise every record's source and score it against its references (ROUGE-1/2/L F1)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import summary_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        hypotheses, truncated, settings = [], 0, None
        for record in checked:
            result = self.summarize(record["source"], **generation)
            hypotheses.append(result["summary"])
            truncated += result["truncated"]
            settings = result["generation"]
        metrics = summary_metrics(hypotheses, [r["targets"] for r in checked])
        metrics.update(
            {
                "hit_token_ceiling": truncated,
                "generation": settings,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _trainable_names(self, trainable_decoder_layers: int) -> list[str]:
        if (
            not isinstance(trainable_decoder_layers, int)
            or not 1 <= trainable_decoder_layers <= DECODER_LAYERS
        ):
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        first = DECODER_LAYERS - trainable_decoder_layers
        prefixes = tuple(f"model.decoder.layers.{k}." for k in range(first, DECODER_LAYERS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 2,
        lr: float = 3e-5,
        batch_size: int = 8,
        trainable_decoder_layers: int = DEFAULT_TRAINABLE_DECODER_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
        eval_generation: Mapping[str, Any] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning on a validated summarization dataset.

        Only the last `trainable_decoder_layers` decoder blocks train (2 by default; the encoder, the shared
        embeddings, the earlier decoder blocks and the tied output projection stay frozen). Teacher-forced
        cross-entropy on the first reference summary, AdamW at a fixed learning rate with gradient clipping
        at 1.0, sources truncated to MAX_TRAIN_SOURCE_TOKENS and targets to MAX_TRAIN_TARGET_TOKENS
        **during training only**. Epoch 0 records the frozen model's validation ROUGE; every epoch is scored
        on the validation split with `eval_generation` (the pipeline defaults unless given), and the epoch
        with the highest validation ROUGE-L is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        names = self._trainable_names(trainable_decoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        generation = dict(eval_generation or {})
        import torch

        torch.manual_seed(seed)
        model, tokenizer = self._require_model()
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = torch.device(self.device)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                k: v
                for k, v in self.evaluate(val_checked, **generation).items()
                if k in ("rouge1", "rouge2", "rougeL", "n", "mean_summary_words", "hit_token_ceiling")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_rouge = entry["val"]["rougeL"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        for epoch in range(1, epochs + 1):
            model.train()
            order = torch.randperm(len(train_checked), generator=generator).tolist()
            losses = []
            for start in range(0, len(order), batch_size):
                batch = [train_checked[i] for i in order[start : start + batch_size]]
                encoded = tokenizer(
                    [r["source"] for r in batch],
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=MAX_TRAIN_SOURCE_TOKENS,
                )
                labels = tokenizer(
                    text_target=[r["targets"][0] for r in batch],
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=MAX_TRAIN_TARGET_TOKENS,
                )["input_ids"]
                labels[labels == tokenizer.pad_token_id] = -100
                out = model(
                    input_ids=encoded["input_ids"].to(device),
                    attention_mask=encoded["attention_mask"].to(device),
                    labels=labels.to(device),
                )
                optimiser.zero_grad(set_to_none=True)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimiser.step()
                losses.append(float(out.loss.detach()))
            model.eval()
            entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
            history.append(entry)
            if progress:
                progress(entry)
            current = entry["val"]["rougeL"] if entry["val"] else math.inf
            if current > best_rouge or not entry["val"]:
                best_rouge = current
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                best_epoch = epoch
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_decoder_layers": trainable_decoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation ROUGE-L" if val_checked else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "max_train_source_tokens": MAX_TRAIN_SOURCE_TOKENS,
            "max_train_target_tokens": MAX_TRAIN_TARGET_TOKENS,
            "eval_generation": generation,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted decoder tensors as safetensors with a manifest naming the pinned base."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest and digest, then overwrite exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        entry = manifest["files"][0]
        weights_path = root / entry["path"]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != manifest["tensors"]:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("model.decoder.layers."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable decoder tensor of the base model"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BARTSummarizationPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/bart_summarization_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Summarization dataset contract for fine-tuning: the pinned SciTLDR sample, validation, seeded
splitting, BYOD loaders and CSV export.

The default dataset is **real** and deliberately out of domain for a CNN/DailyMail news summariser:
SciTLDR (Cachola et al., EMNLP Findings 2020; Apache-2.0), one-sentence "TL;DR" summaries of computer-science
paper abstracts. The `SciTLDR-A` release ships three JSON-Lines files (author-written TLDRs for training,
author plus peer-review-derived TLDRs for dev and test) which are fetched one by one from the project
repository at a pinned commit and refused on any byte-size or SHA-256 mismatch. The frozen model produces
three-sentence, news-style, largely extractive summaries here; the fine-tuning question is whether a bounded
adaptation of the last decoder blocks moves it to the short TL;DR register on held-out papers.

A record is ``{id, source, targets}``: the abstract (sentences joined by a space) and one or more reference
summaries. Training uses the first target; scoring takes the best ROUGE over all of them.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TEXT_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "SciTLDR-A"
CORPUS_RELEASE = "allenai/scitldr @ 5ccad9c00a60ad75c9e04abf7f27d0f53f983b20"
CORPUS_BASE_URL = "https://raw.githubusercontent.com/allenai/scitldr/5ccad9c00a60ad75c9e04abf7f27d0f53f983b20/SciTLDR-Data/SciTLDR-A/"
CORPUS_FILES = {
    "train": ("train.jsonl", 3_155_015, "b222771d387be585cfdf5ae957b36757138415a352e0a3e3b23f73f87c3b1119"),
    "dev": ("dev.jsonl", 1_124_865, "3191fa98ccc09521332b7a1cd63b1930be4e8df125a235ccd31e40329709525e"),
    "test": ("test.jsonl", 1_204_107, "fb42dd6cd4f4a1928ae8a01a189456fbfe994a07e938bd49f68653933f6503c9"),
}
CORPUS_LICENSE = "Apache-2.0 (Cachola et al. 2020; allenai/scitldr)"
CORPUS_PAPERS = {"train": 1_992, "dev": 619, "test": 618}
DEFAULT_CACHE_DIR = Path("weights") / "scitldr"
MAX_SAMPLE_SOURCE_CHARS = (
    2_400  # abstracts above this are left out of the sample (the ceiling is 1,024 tokens)
)
MIN_SAMPLE_SOURCE_CHARS = 200
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 300, "validation": 50, "test": 100}
MIN_RECORDS = 8
MAX_RECORDS = 20_000
MAX_TARGET_CHARS = 2_000
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return the three pinned SciTLDR-A files (bytes) from the cache or the project repository, verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for split, (name, size, digest) in CORPUS_FILES.items():
        local = cache / name
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = CORPUS_BASE_URL + name
            if fetcher is not None:
                data = fetcher(url)
            else:
                with urllib.request.urlopen(url, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{name}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                    f"pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[split] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> dict[str, list[dict[str, Any]]]:
    """Parse the JSON-Lines members into flat records; every paper keeps its SciTLDR `paper_id`."""
    out = {}
    for split in CORPUS_FILES:
        if split not in files:
            raise ValueError(f"corpus is missing the {split} file")
        records = []
        for line in files[split].decode("utf-8").splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            targets = (
                [str(t).strip() for t in row["target"]]
                if isinstance(row["target"], list)
                else [str(row["target"])]
            )
            records.append(
                {
                    "id": f"{split}-{row['paper_id']}",
                    "source": " ".join(str(s).strip() for s in row["source"]),
                    "targets": [t for t in targets if t],
                    "paper_id": str(row["paper_id"]),
                }
            )
        if len(records) != CORPUS_PAPERS[split]:
            raise ValueError(f"{split}: {len(records)} papers, expected {CORPUS_PAPERS[split]}")
        out[split] = records
    return out


def filter_records(records: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Keep records whose source is within the sample length window and that have at least one non-empty
    target; drop repeated sources case-insensitively."""
    seen: set[str] = set()
    kept = []
    for record in records:
        source = str(record["source"])
        if not MIN_SAMPLE_SOURCE_CHARS <= len(source) <= MAX_SAMPLE_SOURCE_CHARS or not record.get("targets"):
            continue
        key = source.lower()
        if key in seen:
            continue
        seen.add(key)
        kept.append(dict(record))
    return kept


def build_sample_dataset(
    corpus: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded draws from the three SciTLDR members: training from `train`, validation from `dev`, test from
    `test` — the release's own paper-disjoint partition, re-checked on sources by `check_split_disjoint`."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    source_of = {"train": "train", "validation": "dev", "test": "test"}
    rng = random.Random(seed)
    out: dict[str, list[dict[str, Any]]] = {}
    for name, size in sizes.items():
        pool = filter_records(corpus[source_of[name]])
        if size > len(pool):
            raise ValueError(f"requested {size} {name} records but only {len(pool)} fit")
        rng.shuffle(pool)
        out[name] = [
            {
                "id": f"{name}-{i:04d}",
                "source": r["source"],
                "targets": list(r["targets"]),
                "paper_id": r["paper_id"],
            }
            for i, r in enumerate(pool[:size])
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/source/targets")
    if "targets" not in record and "target" in record:
        record = {**record, "targets": [record["target"]]}
    for key in ("id", "source", "targets"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid, source, targets = record["id"], record["source"], record["targets"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    if not isinstance(source, str):
        raise ValueError(f"{label}: source must be a string")
    if not source.strip():
        raise ValueError(f"{label}: source is empty")
    if len(source) > MAX_TEXT_CHARS:
        raise ValueError(
            f"{label}: source has {len(source)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}"
        )
    if isinstance(targets, str) or not isinstance(targets, Sequence) or not targets:
        raise ValueError(f"{label}: targets must be a non-empty list of reference summaries")
    checked_targets = []
    for j, target in enumerate(targets):
        if not isinstance(target, str) or not target.strip():
            raise ValueError(f"{label}: targets[{j}] must be a non-empty string")
        if len(target) > MAX_TARGET_CHARS:
            raise ValueError(
                f"{label}: targets[{j}] has {len(target)} chars; "
                f"ceiling is MAX_TARGET_CHARS={MAX_TARGET_CHARS}"
            )
        checked_targets.append(target.strip())
    item = {"id": rid, "source": source.strip(), "targets": checked_targets}
    if "paper_id" in record:
        item["paper_id"] = str(record["paper_id"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a summarization dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, source, targets} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    sources: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        sources.add(item["source"].lower())
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_sources": len(sources),
        "references_per_record": {
            "min": min(len(r["targets"]) for r in checked),
            "max": max(len(r["targets"]) for r in checked),
        },
        "source_chars": {
            "min": min(len(r["source"]) for r in checked),
            "max": max(len(r["source"]) for r in checked),
        },
        "target_words": {
            "min": min(len(r["targets"][0].split()) for r in checked),
            "max": max(len(r["targets"][0].split()) for r in checked),
        },
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["source"], list(r["targets"])] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no lower-cased source document appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["source"]).lower()
            if key in seen and seen[key] != name:
                raise ValueError(
                    f"a document ({record['source'][:60]!r}…) appears in both {seen[key]} and {name}"
                )
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating sources."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = record["source"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {
        "test": unique[:n_test],
        "validation": unique[n_test : n_test + n_val],
        "train": unique[n_test + n_val :],
    }
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a CSV (columns id, source, target — one reference per row), a JSON array or JSONL
    of ``{id, source, target}`` or ``{id, source, targets: [...]}`` objects."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".csv":
        rows = list(csv.DictReader(io.StringIO(text)))
        missing = {"id", "source", "target"} - set(rows[0].keys() if rows else set())
        if missing:
            raise ValueError(f"CSV is missing columns {sorted(missing)}")
        return [{"id": r["id"], "source": r["source"], "targets": [r["target"]]} for r in rows]
    if suffix == ".jsonl":
        return [_normalise(json.loads(line)) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return [_normalise(r) for r in data]
    raise ValueError("BYOD datasets must be .csv, .json or .jsonl")


def _normalise(record: Any) -> Any:
    if isinstance(record, Mapping) and "targets" not in record and "target" in record:
        return {**{k: v for k, v in record.items() if k != "target"}, "targets": [record["target"]]}
    return record


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One row per record with its first reference summary."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "source", "target"])
        writer.writeheader()
        for record in records:
            writer.writerow({"id": record["id"], "source": record["source"], "target": record["targets"][0]})
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `37f520fa929c…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BARTSummarizationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "bart-large-cnn",
  "modelId": "facebook/bart-large-cnn",
  "revision": "37f520fa929c961707657b28798b30c003dd100b",
  "files": [
    {
      "path": "README.md",
      "bytes": 6009,
      "sha256": "a49b4b7fa5e64a05277d57cb767d9b8e8d4789a09cab9647beadceaaed86e170"
    },
    {
      "path": "config.json",
      "bytes": 1585,
      "sha256": "c6cb642aec929b65f514ee0ec7c04f9de19f705c143491577ecd8b7cc923c6ed"
    },
    {
      "path": "generation_config.json",
      "bytes": 363,
      "sha256": "4897361917410254e3132e8fe7786d37f3ef7cff54a650845c1147c2450a790f"
    },
    {
      "path": "generation_config_for_summarization.json",
      "bytes": 363,
      "sha256": "4897361917410254e3132e8fe7786d37f3ef7cff54a650845c1147c2450a790f"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1625222120,
      "sha256": "40041830399afb5348525ef8354b007ecec4286fdf3524f7e6b54377e17096cb"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1355863,
      "sha256": "847bbeab6174d66a88898f729d52fa8d355fafe1bea101cf960dd404581df70e"
    },
    {
      "path": "vocab.json",
      "bytes": 898823,
      "sha256": "9e7f63c2d15d666b52e21d250d2e513b87c9b713cfa6987a82ed89e5e6e50655"
    }
  ],
  "totalBytes": 1627941444
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BARTSummarizationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Referenced corpus, validation and split

`fetch_corpus` downloads the three pinned SciTLDR-A files (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is parsed, and `read_corpus` flattens each JSON-Lines member into records whose `source` is the abstract's sentences joined by a space and whose `targets` are its reference TL;DRs (one author-written TL;DR for training papers; author plus peer-review-derived TL;DRs for dev and test papers). `build_sample_dataset` keeps abstracts of 200..2,400 characters with at least one reference, drops repeated sources, and draws 300 training records from the `train` member, 50 validation records from `dev` and 100 test records from `test` by a seeded shuffle — the release's own paper-disjoint partition. `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no source appears in two splits, and the training split is written to `outputs/bart_summarization_train.csv` in the shape BYOD expects.

Look for: 1,992 + 619 + 618 raw papers, three digests, splits 300 / 50 / 100, one reference per training record and up to four per test record, and four refusal probes — a duplicate id, an empty reference list, a missing field and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_papers = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/scitldr'))
    raw_papers = {name: len(part) for name, part in corpus.items()}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
write_dataset_csv(train_records, 'outputs/bart_summarization_train.csv')
print({'data_source': data_source, 'raw_papers': raw_papers, 'splits': disjoint, 'file_sha256': {k: v[2][:12] + '...' for k, v in CORPUS_FILES.items()}})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_sources': manifest['unique_sources'], 'references_per_record': manifest['references_per_record'], 'source_chars': manifest['source_chars'], 'target_words': manifest['target_words'], 'digest': manifest['digest'][:16] + '...'}})
print({'example': {'id': train_records[0]['id'], 'source': train_records[0]['source'][:200] + '...', 'targets': train_records[0]['targets']}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty reference list': [{**train_records[0], 'targets': []}, *train_records[1:8]],
    'missing field': [{'id': r['id'], 'source': r['source']} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Summarise through the inference contract

Before any adaptation, the inference contract is exercised as it always was, on a synthetic three-paragraph news-style document authored in this cell (the model card's smoke passage). `validate_inputs` applies exactly the checks `summarize` applies — text type and character ceiling, and the generation settings against their ceilings — and returns an input manifest; the encoder-token ceiling `MAX_INPUT_TOKENS` (1,024) needs the real tokenizer and is enforced inside `summarize`, which **rejects with a `ValueError` naming the count, never truncates or chunks**. An out-of-range `num_beams` is validated too and its rejection recorded as a finding. `summarize` returns the summary with `generated_tokens`, `input_tokens`, `truncated` (the summary hit `max_new_tokens`) and `stopped_by`, and echoes the settings. **Score semantics:** the pipeline emits **no probability, confidence or score of any kind**; `truncated` is a length flag. The first call uses the pinned news defaults (at least 55 new tokens); the second uses the TL;DR-length settings every ROUGE number below is produced under, so the length difference is visible before any metric is read.

In [ ]:
import time

TLDR_MAX_NEW_TOKENS = 48  # @param {type:"integer"}
TLDR_MIN_NEW_TOKENS = 0  # @param {type:"integer"}
NUM_BEAMS = 4  # @param {type:"integer"}

TLDR = {'max_new_tokens': TLDR_MAX_NEW_TOKENS, 'min_new_tokens': TLDR_MIN_NEW_TOKENS, 'num_beams': NUM_BEAMS}
document = (
    'The town council of Millbrook voted on Tuesday evening to approve a three-year plan to replace the '
    'aging water mains beneath the historic district. The plan, which had been debated for more than a '
    'year, will cost an estimated 4.2 million dollars and is scheduled to begin in the spring. Council '
    'members said the decision was driven by a series of pipe failures last winter that left several '
    'streets without water for days.\n\n'
    'Under the approved schedule, crews will work one block at a time so that no more than two streets are '
    'closed on any given day. The public works director told residents that most of the disruption would '
    'fall in the first eighteen months, with paving and landscaping to follow. Businesses along Main Street '
    'will receive advance notice of closures and a dedicated contact for complaints.\n\n'
    'Funding will come from a combination of a state infrastructure grant and a modest increase in water '
    'rates, which the council set at three percent per year for the duration of the project. Two members '
    'voted against the rate increase, arguing that the grant alone should have covered the work, but the '
    'majority said delaying the project any further would only raise its cost.'
)
document_id = 'doc'
ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_INPUT_TOKENS': MAX_INPUT_TOKENS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'MAX_NUM_BEAMS': MAX_NUM_BEAMS, 'LENGTH_PENALTY_RANGE': LENGTH_PENALTY_RANGE, 'MAX_NO_REPEAT_NGRAM_SIZE': MAX_NO_REPEAT_NGRAM_SIZE}
print(ceilings)
print({'defaults_from': GENERATION_CONFIG_FILE, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DEFAULT_MIN_NEW_TOKENS': DEFAULT_MIN_NEW_TOKENS, 'DEFAULT_NUM_BEAMS': DEFAULT_NUM_BEAMS, 'DEFAULT_LENGTH_PENALTY': DEFAULT_LENGTH_PENALTY, 'DEFAULT_NO_REPEAT_NGRAM_SIZE': DEFAULT_NO_REPEAT_NGRAM_SIZE, 'EARLY_STOPPING': EARLY_STOPPING, 'DECISION_RULE': DECISION_RULE})
input_manifest = validate_inputs([document], num_beams=NUM_BEAMS, names=[document_id])
try:
    validate_inputs([document], num_beams=MAX_NUM_BEAMS + 1)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'num-beams-ceiling-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/bart_summarization_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
started = time.perf_counter()
result = pipe.summarize(document, num_beams=NUM_BEAMS)
summarize_elapsed = time.perf_counter() - started
checks = {
    'summary_is_non_empty_text': isinstance(result['summary'], str) and bool(result['summary'].strip()),
    'generated_within_bound': 1 <= result['generated_tokens'] <= DEFAULT_MAX_NEW_TOKENS <= MAX_NEW_TOKENS,
    'input_within_ceiling': 1 <= result['input_tokens'] <= MAX_INPUT_TOKENS,
    'truncated_matches_stopped_by': result['truncated'] == (result['stopped_by'] != 'eos'),
    'settings_echoed': result['generation']['max_new_tokens'] == DEFAULT_MAX_NEW_TOKENS and result['generation']['num_beams'] == NUM_BEAMS,
    'deterministic_decoding': result['generation']['do_sample'] is False,
}
if not all(checks.values()):
    raise RuntimeError(f'summarize output failed a sanity check: {checks}')
print({key: value for key, value in result.items() if key != 'summary'})
print({'seconds': round(summarize_elapsed, 3), 'checks': checks, 'no_score': 'the pipeline emits no probability or quality score; truncated is a length flag', 'findings': len(input_manifest['findings'])})
print('summary (pinned news defaults):')
print(result['summary'])
started = time.perf_counter()
tldr_result = pipe.summarize(document, **TLDR)
print({'tldr_settings': TLDR, 'generated_tokens': tldr_result['generated_tokens'], 'stopped_by': tldr_result['stopped_by'], 'seconds': round(time.perf_counter() - started, 3)})
print('summary (TL;DR-length settings):')
print(tldr_result['summary'])

## 6. Baselines and the frozen model's score on the test split

Three numbers frame the adaptation, all under the TL;DR-length settings of Section 5. The **Lead-1 baseline** submits the first sentence of each abstract as its summary and the **Lead-3 baseline** the first three: what a system that does no modelling gets, and a reminder that ROUGE rewards length matching — Lead-1 usually beats Lead-3 against one-sentence references. The **frozen model** summarises the 100 test abstracts and is scored with the same three metrics: corpus **ROUGE-1**, **ROUGE-2** and **ROUGE-L** F1 (best over the references, rouge-score-style, not rouge-score-identical). Expect the frozen model to land near the Lead baselines with summaries roughly twice as long as the references — it copies the abstract's opening sentences in the news register it was trained for — and read `mean_summary_words` and `hit_token_ceiling` (summaries cut at `max_new_tokens`) before trusting any score. About three minutes on CPU.

In [ ]:
baseline_lead1 = lead_baseline(test_records, n_sentences=1)
baseline_lead3 = lead_baseline(test_records, n_sentences=3)
for name, baseline in (('lead1_baseline', baseline_lead1), ('lead3_baseline', baseline_lead3)):
    print({name: {'rouge1': round(baseline['rouge1'], 2), 'rouge2': round(baseline['rouge2'], 2), 'rougeL': round(baseline['rougeL'], 2), 'mean_summary_words': round(baseline['mean_summary_words'], 1), 'n': baseline['n']}})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, **TLDR)
print({'frozen_model_test': {'rouge1': round(frozen_test['rouge1'], 2), 'rouge2': round(frozen_test['rouge2'], 2), 'rougeL': round(frozen_test['rougeL'], 2), 'mean_summary_words': round(frozen_test['mean_summary_words'], 1), 'mean_reference_words': round(frozen_test['mean_reference_words'], 1), 'hit_token_ceiling': frozen_test['hit_token_ceiling'], 'n': frozen_test['n'], 'verdict': frozen_test['verdict']}, 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
for record in test_records[:2]:
    print({'frozen': pipe.summarize(record['source'], **TLDR)['summary'], 'reference': record['targets'][0]})
assert frozen_test['rouge1'] > 0.0

## 7. Bounded fine-tuning of the last decoder blocks

`pipe.adapt` trains only the last `TRAINABLE_DECODER_LAYERS` decoder blocks — two by default, 33,593,344 of 406,290,432 parameters; the encoder, the shared embeddings, the tied output projection and the earlier decoder blocks stay frozen — with teacher-forced cross-entropy on the first reference TL;DR, AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Sources are truncated to 512 and targets to 64 BPE tokens **during training only**. Epoch 0 records the frozen model's validation ROUGE under the TL;DR settings; every epoch is scored on the validation split the same way, and the epoch with the highest validation ROUGE-L is kept.

Watch validation ROUGE-L rise by several points and `mean_summary_words` fall towards the reference length within the first epoch (about a minute of training plus a validation pass per epoch on CPU). The build record's counter-example: one epoch over 200 abstracts already captured most of the gain.

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 3e-5  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_DECODER_LAYERS = 2  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_rouge1'] = round(entry['val']['rouge1'], 2)
        row['val_rouge2'] = round(entry['val']['rouge2'], 2)
        row['val_rougeL'] = round(entry['val']['rougeL'], 2)
        row['val_mean_summary_words'] = round(entry['val']['mean_summary_words'], 1)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_decoder_layers=TRAINABLE_DECODER_LAYERS, eval_generation=TLDR, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or epoch selection, and no abstract in it appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6, and the four numbers are put side by side. Look for a ROUGE-L gain of several points over the frozen model and over both Lead baselines, and for `mean_summary_words` close to the reference length — the cell asserts the adapted ROUGE-L is above the frozen ROUGE-L — and for the same two abstracts summarised by the adapted model. One hundred abstracts from one seeded split of one corpus give no dispersion estimate; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a register shift on paper abstracts says nothing about your documents until you measure it there. Note what the adaptation also changes: the model now writes one short sentence, so its long news-highlight behaviour is traded away in the adapter.

In [ ]:
adapted_test = pipe.evaluate(test_records, **TLDR)
adapted_val = pipe.evaluate(val_records, **TLDR)
comparison = {
    metric: {'lead1': round(baseline_lead1[metric], 2), 'lead3': round(baseline_lead3[metric], 2), 'frozen': round(frozen_test[metric], 2), 'adapted': round(adapted_test[metric], 2)}
    for metric in ('rouge1', 'rouge2', 'rougeL', 'mean_summary_words')
}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 2) for metric in ('rouge1', 'rouge2', 'rougeL')}
for metric, row in comparison.items():
    print({metric: row})
for record in test_records[:2]:
    print({'adapted': pipe.summarize(record['source'], **TLDR)['summary'], 'reference': record['targets'][0]})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'generation': frozen_test['generation'],
    'baselines': {'lead1': baseline_lead1, 'lead3': baseline_lead3},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/bart_summarization_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['rougeL'] > frozen_test['rougeL']
print({'report': 'outputs/bart_summarization_evaluation_report.json'})

## 9. Summarise new abstracts, export the adapter and reload it

Four abstracts that were in none of the splits are summarised by the adapted model through the same `summarize` contract as Section 5 and scored with `pipe.evaluate` (a `measured-small-sample` verdict, because four documents carry no dispersion estimate); the single-document `evaluation_report` helper — the inference-stage helper, which now scores supplied references as `sample-sanity` — is written for the first of them.

`pipe.save_artifact` writes the trained tensors — the last two decoder blocks, about 134 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `BARTSummarizationPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not an adaptable decoder tensor, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical summaries (VER4).

In [ ]:
import csv
import shutil

if USE_BYOD:
    new_records = [{**r, 'id': f'new-{i:02d}'} for i, r in enumerate(test_records[:4])]
else:
    used = {r['source'].lower() for part in splits.values() for r in part}
    new_records = [{**r, 'id': f'new-{i:02d}'} for i, r in enumerate([r for r in filter_records(corpus['dev']) if r['source'].lower() not in used][:4])]
new_metrics = pipe.evaluate(new_records, **TLDR)
new_results = []
for record in new_records:
    item = pipe.summarize(record['source'], **TLDR)
    new_results.append({'id': record['id'], 'summary': item['summary'], 'reference': record['targets'][0], 'generated_tokens': item['generated_tokens'], 'input_tokens': item['input_tokens'], 'truncated': item['truncated'], 'stopped_by': item['stopped_by']})
    print({k: new_results[-1][k] for k in ('id', 'summary', 'reference')})
single_report = evaluation_report(pipe.summarize(new_records[0]['source'], **TLDR), new_records[0]['targets'], sample_kind='one unseen SciTLDR abstract' if not USE_BYOD else 'one BYOD test record')
print({'new_abstracts': {'n': new_metrics['n'], 'rouge1': round(new_metrics['rouge1'], 2), 'rougeL': round(new_metrics['rougeL'], 2), 'verdict': new_metrics['verdict']}, 'single_document_report_verdict': single_report['verdict']})
with open('outputs/bart_summarization_summaries.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(new_results[0]))
    writer.writeheader()
    writer.writerows(new_results)

artifact_dir = Path('outputs/bart_summarization_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'bart_summarization', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = BARTSummarizationPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [pipe.summarize(r['source'], **TLDR)['summary'] for r in test_records[:4]]
after = [reloaded.summarize(r['source'], **TLDR)['summary'] for r in test_records[:4]]
parity = {'identical_summaries': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_summaries'] == parity['of']

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'files': {k: {'name': v[0], 'bytes': v[1], 'sha256': v[2]} for k, v in CORPUS_FILES.items()}, 'license': CORPUS_LICENSE},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'pinned_defaults_result': {k: v for k, v in result.items() if k != 'summary'}, 'tldr_settings': TLDR},
    'comparison': comparison,
    'new_abstracts': new_metrics,
    'single_document_report': single_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/bart_summarization_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen news summariser, asked for a TL;DR of a paper abstract, writes something close to the abstract's own opening sentences at twice the reference length and scores near the Lead baselines; a bounded fine-tuning of the last two decoder blocks on 300 in-domain abstracts moves it to the one-sentence register and lifts held-out ROUGE-L by several points in a few minutes on CPU, with a 134 MB adapter that reloads to identical summaries. That is the claim: the adaptation contract works end to end on a real referenced corpus, and the numbers it produces are read against two Lead baselines and the frozen model rather than in isolation.

The test split is 100 abstracts from one seeded split of one corpus, the metrics are three n-gram overlap scores (own implementation, not rouge-score-identical, and none a judgement of faithfulness), and SciTLDR references are short and formulaic. So a gain here says the contract works, not that the adapted model is better on your documents, that it handles long or technical text, or that its summaries are faithful — an abstractive summary can state a result the source does not and still overlap the reference well. Fine-tuning on a narrow corpus also changes the model elsewhere — the adapter writes one short sentence for any input — and nothing here measures that.

Three things to carry to real data. **References first:** the Lead baselines and the frozen model's score on *your* references, under *your* length settings, are the numbers to read before any adapted one — ROUGE moves with summary length as much as with content. **Leakage:** de-duplicate sources across splits (the contract does this case-insensitively) and split by document collection or author when your pairs come from one. **Ceilings:** inputs over `MAX_INPUT_TOKENS` are refused at inference and truncated to 512 tokens only during training — long-document summarization is out of scope.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real referenced corpus, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against two trivial baselines and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, summary quality or faithfulness on any other domain, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_DECODER_LAYERS = 1` and compare the artifact size and the test scores; raise `TLDR_MAX_NEW_TOKENS` and watch ROUGE fall as summaries lengthen; set `NUM_BEAMS = 1` and read the greedy scores; or bring your own document–summary pairs through BYOD and read the Lead baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/bart-cnn-summarization-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/bart-cnn-summarization-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/bart-cnn-summarization-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/facebook/bart-large-cnn
- Upstream code: https://github.com/facebookresearch/fairseq/tree/main/examples/bart
- BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension (Lewis et al., 2019): https://arxiv.org/abs/1910.13461
- TLDR: Extreme Summarization of Scientific Documents (Cachola et al., EMNLP Findings 2020; SciTLDR, Apache-2.0): https://arxiv.org/abs/2004.15011
- ROUGE: A Package for Automatic Evaluation of Summaries (Lin, 2004): https://aclanthology.org/W04-1013
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)